# 04 — Forecasting Models
### PM Accelerator — Weather Trend Forecasting Assessment

Builds and evaluates three forecasting models + ensemble:
- **Model 1:** ARIMA (2,1,2)
- **Model 2:** Facebook Prophet
- **Model 3:** XGBoost with lag features
- **Ensemble:** Simple average of all three

**Target:** daily mean `temperature_celsius` (global average)  
**Split:** 80% train / 20% test — chronological  
**Metrics:** MAE, RMSE, MAPE

In [5]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from src.utils import (
    DATA_DIR, FIGURES_DIR, MODELS_DIR, OUTPUT_DIR,
    compute_mae, compute_rmse, compute_mape,
    apply_plot_style, save_figure,
)

apply_plot_style()
%matplotlib inline

## Load & Prepare Daily Series

In [6]:
df = pd.read_csv(
    os.path.join(DATA_DIR, 'cleaned_weather_unscaled.csv'),
    parse_dates=['last_updated'],
)
df.set_index('last_updated', inplace=True)
df.sort_index(inplace=True)

daily = df['temperature_celsius'].resample('D').mean().dropna()
print(f'Daily series: {len(daily)} days')
print(f'Date range: {daily.index.min()} → {daily.index.max()}')

Daily series: 674 days
Date range: 2024-05-16 00:00:00 → 2026-03-21 00:00:00


## Train / Test Split (80/20 Chronological)

In [7]:
split_idx = int(len(daily) * 0.8)
train = daily.iloc[:split_idx]
test = daily.iloc[split_idx:]
print(f'Train: {len(train)} days ({train.index.min()} → {train.index.max()})')
print(f'Test:  {len(test)} days ({test.index.min()} → {test.index.max()})')

predictions = {}

Train: 539 days (2024-05-16 00:00:00 → 2025-11-06 00:00:00)
Test:  135 days (2025-11-07 00:00:00 → 2026-03-21 00:00:00)


In [8]:
pip install statsmodels

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Model 1 — ARIMA(2,1,2)

In [9]:
from statsmodels.tsa.arima.model import ARIMA

try:
    model_arima = ARIMA(train, order=(2, 1, 2))
    result_arima = model_arima.fit()
    pred_arima = result_arima.forecast(steps=len(test))
    pred_arima.index = test.index
    predictions['ARIMA'] = pred_arima.values

    print(f'MAE:  {compute_mae(test, pred_arima):.4f}')
    print(f'RMSE: {compute_rmse(test, pred_arima):.4f}')
    print(f'MAPE: {compute_mape(test, pred_arima):.4f}%')
    joblib.dump(result_arima, os.path.join(MODELS_DIR, 'arima_model.pkl'))
except Exception as e:
    print(f'ARIMA failed: {e}')
    predictions['ARIMA'] = np.full(len(test), train.mean())

MAE:  3.5493
RMSE: 3.9109
MAPE: 24.8147%


In [10]:
pip install prophet 2>&1


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Model 2 — Facebook Prophet

In [11]:
try:
    from prophet import Prophet

    train_prophet = pd.DataFrame({'ds': train.index, 'y': train.values})
    m = Prophet(yearly_seasonality=True, weekly_seasonality=True)
    m.fit(train_prophet)

    future = m.make_future_dataframe(periods=len(test))
    forecast = m.predict(future)

    pred_prophet_full = forecast.set_index('ds')['yhat']
    pred_prophet = pred_prophet_full.reindex(test.index).values
    if np.any(np.isnan(pred_prophet)):
        pred_prophet = forecast['yhat'].values[-len(test):]

    predictions['Prophet'] = pred_prophet
    print(f'MAE:  {compute_mae(test.values, pred_prophet):.4f}')
    print(f'RMSE: {compute_rmse(test.values, pred_prophet):.4f}')
    print(f'MAPE: {compute_mape(test.values, pred_prophet):.4f}%')
    joblib.dump(m, os.path.join(MODELS_DIR, 'prophet_model.pkl'))

except Exception as e:
    print(f'Prophet failed: {e}')
    print('Falling back to SARIMA...')
    try:
        from statsmodels.tsa.statespace.sarimax import SARIMAX
        model_sarima = SARIMAX(train, order=(2, 1, 2),
                               seasonal_order=(1, 1, 1, 7),
                               enforce_stationarity=False,
                               enforce_invertibility=False)
        result_sarima = model_sarima.fit(disp=False)
        pred_sarima = result_sarima.forecast(steps=len(test))
        predictions['Prophet (SARIMA fallback)'] = pred_sarima.values
        print(f'MAE:  {compute_mae(test, pred_sarima):.4f}')
        print(f'RMSE: {compute_rmse(test, pred_sarima):.4f}')
        print(f'MAPE: {compute_mape(test, pred_sarima):.4f}%')
    except Exception as e2:
        print(f'SARIMA also failed: {e2}')
        predictions['Prophet'] = np.full(len(test), train.mean())

02:54:02 - cmdstanpy - INFO - Chain [1] start processing
02:54:03 - cmdstanpy - INFO - Chain [1] done processing


MAE:  1.0060
RMSE: 1.5798
MAPE: 8.1736%


## Model 3 — XGBoost with Lag Features

In [12]:
import xgboost as xgb

# Create lag features
df_xgb = pd.DataFrame({'temperature_celsius': daily})
for lag in [1, 3, 7, 14, 30]:
    df_xgb[f'temp_lag_{lag}'] = df_xgb['temperature_celsius'].shift(lag)
df_xgb['rolling_mean_7'] = df_xgb['temperature_celsius'].rolling(7).mean()
df_xgb['rolling_std_7'] = df_xgb['temperature_celsius'].rolling(7).std()
df_xgb['day_of_year'] = df_xgb.index.dayofyear
df_xgb['month'] = df_xgb.index.month
df_xgb.dropna(inplace=True)

feature_cols = [c for c in df_xgb.columns if c != 'temperature_celsius']
X_all = df_xgb[feature_cols]
y_all = df_xgb['temperature_celsius']

# Chronological split
split_date = test.index[0]
X_train_xgb = X_all[X_all.index < split_date]
X_test_xgb = X_all[X_all.index >= split_date]
y_train_xgb = y_all[y_all.index < split_date]
y_test_xgb = y_all[y_all.index >= split_date]

print(f'Training samples: {len(X_train_xgb)}')
print(f'Test samples:     {len(X_test_xgb)}')

model_xgb = xgb.XGBRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=6,
    random_state=42, n_jobs=-1,
)
model_xgb.fit(X_train_xgb, y_train_xgb,
              eval_set=[(X_test_xgb, y_test_xgb)], verbose=False)

pred_xgb_full = model_xgb.predict(X_test_xgb)
pred_xgb_series = pd.Series(pred_xgb_full, index=X_test_xgb.index)
pred_xgb_aligned = pred_xgb_series.reindex(test.index).ffill().bfill()
predictions['XGBoost'] = pred_xgb_aligned.values

print(f'MAE:  {compute_mae(test.values, predictions["XGBoost"]):.4f}')
print(f'RMSE: {compute_rmse(test.values, predictions["XGBoost"]):.4f}')
print(f'MAPE: {compute_mape(test.values, predictions["XGBoost"]):.4f}%')
joblib.dump(model_xgb, os.path.join(MODELS_DIR, 'xgboost_model.pkl'))

Training samples: 509
Test samples:     135
MAE:  1.2332
RMSE: 1.8496
MAPE: 9.9384%


['c:\\Users\\purva\\OneDrive\\Desktop\\PMA\\outputs\\models\\xgboost_model.pkl']

## Ensemble — Simple Average

In [13]:
pred_arrays = [v for v in predictions.values()]
min_len = min(len(a) for a in pred_arrays)
pred_arrays_trunc = [a[:min_len] for a in pred_arrays]
test_trunc = test.values[:min_len]

ensemble_pred = np.mean(pred_arrays_trunc, axis=0)
predictions['Ensemble'] = ensemble_pred

print(f'Ensemble MAE:  {compute_mae(test_trunc, ensemble_pred):.4f}')
print(f'Ensemble RMSE: {compute_rmse(test_trunc, ensemble_pred):.4f}')
print(f'Ensemble MAPE: {compute_mape(test_trunc, ensemble_pred):.4f}%')

Ensemble MAE:  1.8674
Ensemble RMSE: 2.3234
Ensemble MAPE: 13.9587%


## Model Comparison Table

In [14]:
comparison_dict = {}
for name, preds in predictions.items():
    preds_t = preds[:min_len]
    comparison_dict[name] = {
        'MAE': round(compute_mae(test_trunc, preds_t), 4),
        'RMSE': round(compute_rmse(test_trunc, preds_t), 4),
        'MAPE (%)': round(compute_mape(test_trunc, preds_t), 4),
    }

comparison_df = pd.DataFrame(comparison_dict).T
comparison_df.index.name = 'Model'
display(comparison_df)

comp_path = os.path.join(OUTPUT_DIR, 'model_comparison.csv')
comparison_df.to_csv(comp_path)
print(f'\nComparison saved → {comp_path}')

,MAE,RMSE,MAPE (%)
Model,,,
ARIMA,3.5493,3.9109,24.8147
Prophet,1.0060,1.5798,8.1736
XGBoost,1.2332,1.8496,9.9384
Ensemble,1.8674,2.3234,13.9587



Comparison saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\model_comparison.csv


## Forecast Plots

In [15]:
# All models comparison
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test.index[:min_len], test_trunc, color='black', linewidth=1.5, label='Actual', alpha=0.8)

colors = {'ARIMA': '#E74C3C', 'Prophet': '#3498DB', 'Prophet (SARIMA fallback)': '#3498DB',
          'XGBoost': '#2ECC71', 'Ensemble': '#9B59B6'}

for name, preds in predictions.items():
    if name == 'Ensemble':
        continue
    ax.plot(test.index[:min_len], preds[:min_len], linewidth=1.0, alpha=0.7,
            label=name, color=colors.get(name, '#95A5A6'))

ax.plot(test.index[:min_len], ensemble_pred, linewidth=2.0, alpha=0.9,
        label='Ensemble', color='#9B59B6', linestyle='--')

ax.set_title('Temperature Forecast — Model Comparison', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
save_figure(fig, '08_forecast_comparison.png')
plt.show()

Figure saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\figures\08_forecast_comparison.png


In [16]:
# Full timeline
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index, train.values, color='#3498DB', linewidth=0.8, alpha=0.6, label='Train')
ax.plot(test.index[:min_len], test_trunc, color='black', linewidth=1.5, label='Actual (Test)')
ax.plot(test.index[:min_len], ensemble_pred, color='#E74C3C', linewidth=1.5,
        linestyle='--', label='Ensemble Forecast')
ax.axvline(x=test.index[0], color='gray', linestyle=':', alpha=0.7)
ax.set_title('Full Timeline — Train, Test & Ensemble Forecast', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
save_figure(fig, '09_full_timeline_forecast.png')
plt.show()

print('\n✅ Forecasting complete!')

Figure saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\figures\09_full_timeline_forecast.png

✅ Forecasting complete!
